# Comparação fuzzy no conjunto de teste

Este notebook resume os resultados salvos em `output/segmentation/runs/mid_res/fuzzy_comparison`.

O foco é comparar:

- detecção dos óstios;
- Dice score da segmentação arterial;
- impacto do fuzzy threshold e do fuzzy connectedness em relação ao baseline.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from utils.project.notebook_env import configure_notebook_environment
    REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
except Exception:
    current = Path.cwd().resolve()
    REPO_ROOT = next(
        path for path in [current, *current.parents]
        if (path / "src").exists() and (path / "output").exists()
    )

RESULT_ROOT = REPO_ROOT / "output/segmentation/runs/mid_res/fuzzy_comparison"
FIGURE_DIR = REPO_ROOT / "output/segmentation/analysis/fuzzy_comparison_eda/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


def save_current_figure(name: str):
    path = FIGURE_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Figura salva em: {path.relative_to(REPO_ROOT)}")

RESULT_ROOT

## Carregamento dos Resultados

A análise usa o `ostios_test_summary.csv` de cada variante como fonte principal, porque esse CSV registra por imagem o método de threshold e o método arterial efetivamente usados.

In [ ]:
SUCCESS_LABELS = {
    "both correct",
    "both tolerable",
    "both ostia correct",
    "both ostia tolerable",
}
CORRECT_LABELS = {"both correct", "both ostia correct"}
TOLERABLE_LABELS = {"both tolerable", "both ostia tolerable"}
WRONG_LABELS = {"found but incorrect", "found_but_wrong"}

PREFERRED_ORDER = [
    "normal_rg",
    "th_fuzzy_rg",
    "normal_fc",
    "th_fuzzy_fc",
]

PRETTY_NAMES = {
    "normal_rg": "Normal + RG",
    "th_fuzzy_rg": "Fuzzy threshold + RG",
    "normal_fc": "Normal + FC",
    "th_fuzzy_fc": "Fuzzy threshold + FC",
}


def yes_no_to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["yes", "true", "1", "sim"])


def first_existing_value(df: pd.DataFrame, columns: list[str], default: str = "") -> str:
    for column in columns:
        if column in df.columns and not df[column].empty:
            value = df[column].iloc[0]
            if pd.notna(value):
                return str(value)
    return default


def load_variant_run(summary_path: Path) -> tuple[pd.DataFrame, dict]:
    variant = summary_path.parents[2].name
    run_dir = summary_path.parents[1]
    df = pd.read_csv(summary_path)
    df["folder_variant"] = variant
    df["variant_label"] = PRETTY_NAMES.get(variant, variant)
    df["run_timestamp"] = run_dir.name
    df["run_dir"] = str(run_dir.relative_to(REPO_ROOT))
    df["artery_dice"] = pd.to_numeric(df["artery_dice"], errors="coerce")
    df["ostia_detected_bool"] = yes_no_to_bool(df["ostia_detected"])
    df["ostia_success"] = df["ostia_detection_status"].astype(str).isin(SUCCESS_LABELS)
    df["both_correct"] = df["ostia_detection_status"].astype(str).isin(CORRECT_LABELS)
    df["both_tolerable"] = df["ostia_detection_status"].astype(str).isin(TOLERABLE_LABELS)
    df["found_wrong"] = df["ostia_detection_status"].astype(str).isin(WRONG_LABELS)

    summary = {
        "folder_variant": variant,
        "variant_label": PRETTY_NAMES.get(variant, variant),
        "run_timestamp": run_dir.name,
        "run_dir": str(run_dir.relative_to(REPO_ROOT)),
        "n_images": len(df),
        "threshold_mode": first_existing_value(df, ["threshold_mode"], "normal"),
        "artery_method": first_existing_value(
            df,
            ["configured_artery_segmentation_method", "artery_segmentation_method"],
            "",
        ),
        "ostia_detected_rate": df["ostia_detected_bool"].mean(),
        "ostia_success_rate": df["ostia_success"].mean(),
        "both_correct_n": int(df["both_correct"].sum()),
        "both_tolerable_n": int(df["both_tolerable"].sum()),
        "found_wrong_n": int(df["found_wrong"].sum()),
        "not_found_or_error_n": int((~(df["ostia_success"] | df["found_wrong"])).sum()),
        "mean_dice": df["artery_dice"].mean(),
        "median_dice": df["artery_dice"].median(),
        "std_dice": df["artery_dice"].std(),
        "mean_dice_success_ostia": df.loc[df["ostia_success"], "artery_dice"].mean(),
    }
    return df, summary

In [ ]:
all_frames = []
summary_rows = []
for summary_path in sorted(RESULT_ROOT.glob("*/*/numeric/ostios_test_summary.csv")):
    variant = summary_path.parents[2].name
    if variant not in PREFERRED_ORDER:
        continue
    df_variant, summary = load_variant_run(summary_path)
    all_frames.append(df_variant)
    summary_rows.append(summary)

if not all_frames:
    raise FileNotFoundError(f"Nenhum ostios_test_summary.csv encontrado em {RESULT_ROOT}")

results_df = pd.concat(all_frames, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
summary_df["order"] = summary_df["folder_variant"].map({name: idx for idx, name in enumerate(PREFERRED_ORDER)})
summary_df = summary_df.sort_values(["order", "folder_variant"]).drop(columns="order")

print(f"Runs carregados: {summary_df.shape[0]}")
print(f"Linhas por imagem: {results_df.shape[0]}")

In [ ]:
display(summary_df)

## Resumo Geral

A tabela abaixo ordena as variantes por sucesso dos óstios e Dice médio. O sucesso dos óstios considera `both correct` ou `both tolerable` como sucesso.

In [ ]:
ranking_df = summary_df.sort_values(
    ["ostia_success_rate", "mean_dice", "mean_dice_success_ostia"],
    ascending=False,
).copy()

ranking_display = ranking_df[[
    "variant_label",
    "threshold_mode",
    "artery_method",
    "n_images",
    "ostia_detected_rate",
    "ostia_success_rate",
    "both_correct_n",
    "both_tolerable_n",
    "found_wrong_n",
    "not_found_or_error_n",
    "mean_dice",
    "median_dice",
    "mean_dice_success_ostia",
]].copy()

for col in ["ostia_detected_rate", "ostia_success_rate"]:
    ranking_display[col] = 100 * ranking_display[col]

In [ ]:
display(ranking_display.round({
    "ostia_detected_rate": 1,
    "ostia_success_rate": 1,
    "mean_dice": 4,
    "median_dice": 4,
    "mean_dice_success_ostia": 4,
}))

## Detecção dos Óstios

Aqui o status é separado em: ambos corretos, ambos toleráveis, encontrados porém incorretos e não encontrado/erro.

In [ ]:
ostia_plot = summary_df.copy()
ostia_plot["variant_label"] = pd.Categorical(
    ostia_plot["variant_label"],
    [PRETTY_NAMES.get(name, name) for name in PREFERRED_ORDER],
    ordered=True,
)
ostia_plot = ostia_plot.sort_values("variant_label")

In [ ]:
status_cols = ["both_correct_n", "both_tolerable_n", "found_wrong_n", "not_found_or_error_n"]
status_labels = ["Both correct", "Both tolerable", "Found but wrong", "Not found/error"]
colors = ["#2ca02c", "#8fd175", "#ff9f1a", "#d62728"]

fig, ax = plt.subplots(figsize=(12, 5))
x_labels = ostia_plot["variant_label"].astype(str)
bottom = np.zeros(len(ostia_plot))
for col, label, color in zip(status_cols, status_labels, colors):
    values = ostia_plot[col].to_numpy()
    bars = ax.bar(x_labels, values, bottom=bottom, label=label, color=color)
    for bar, value, base in zip(bars, values, bottom):
        if value > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                base + value / 2,
                f"{int(value)}",
                ha="center",
                va="center",
                fontsize=9,
                color="black",
            )
    bottom += values

for x_pos, total in zip(range(len(x_labels)), bottom):
    ax.text(x_pos, total + max(bottom) * 0.015, f"{int(total)}", ha="center", va="bottom", fontsize=10)

ax.set_ylabel("Número de imagens", fontsize=12)
ax.set_xlabel("")
ax.set_ylim(0, max(bottom) * 1.12)
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.legend(ncol=2, frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("ostia_status_by_variant.png")
plt.show()

## Dice Score por Variante

A tabela abaixo resume o Dice arterial por variante. O gráfico em seguida mostra média e mediana para comparação visual.

In [ ]:
dice_plot = summary_df.copy()
dice_plot["variant_label"] = pd.Categorical(
    dice_plot["variant_label"],
    [PRETTY_NAMES.get(name, name) for name in PREFERRED_ORDER],
    ordered=True,
)
dice_plot = dice_plot.sort_values("variant_label")

In [ ]:
dice_stats_df = (
    results_df.groupby(["folder_variant", "variant_label"], as_index=False)["artery_dice"]
    .agg(
        mean_dice="mean",
        max_dice="max",
        min_dice="min",
        std_dice="std",
        median_dice="median",
    )
)
dice_stats_df["order"] = dice_stats_df["folder_variant"].map({name: idx for idx, name in enumerate(PREFERRED_ORDER)})
dice_stats_df = dice_stats_df.sort_values(["order", "folder_variant"]).drop(columns="order")

dice_stats_csv_path = FIGURE_DIR.parent / "dice_stats_by_variant.csv"
dice_stats_df.to_csv(dice_stats_csv_path, index=False)
print(f"CSV salvo em: {dice_stats_csv_path.relative_to(REPO_ROOT)}")

display(dice_stats_df.round({
    "mean_dice": 4,
    "max_dice": 4,
    "min_dice": 4,
    "std_dice": 4,
    "median_dice": 4,
}))

In [ ]:
x = np.arange(len(dice_plot))
width = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
mean_bars = ax.bar(x - width/2, dice_plot["mean_dice"], width, label="Média", color="#4c78a8")
median_bars = ax.bar(x + width/2, dice_plot["median_dice"], width, label="Mediana", color="#f58518")

for bars in [mean_bars, median_bars]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.008,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(dice_plot["variant_label"].astype(str), rotation=35, ha="right")
ax.set_ylabel("Dice arterial", fontsize=12)
y_max = max(0.75, float(dice_plot[["mean_dice", "median_dice"]].max().max()) + 0.08)
ax.set_ylim(0, y_max)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("dice_mean_median_by_variant.png")
plt.show()

## Comparação com Baseline

A célula abaixo usa `Normal + RG` como baseline e calcula o ganho/perda de Dice por imagem.

In [ ]:
baseline_variant = "normal_rg"
baseline = results_df.loc[
    results_df["folder_variant"] == baseline_variant,
    ["IMG_ID", "artery_dice", "ostia_detection_status"],
].rename(columns={"artery_dice": "baseline_dice", "ostia_detection_status": "baseline_ostia_status"})

delta_rows = []
for variant in PREFERRED_ORDER:
    if variant == baseline_variant or variant not in set(results_df["folder_variant"]):
        continue
    current = results_df.loc[
        results_df["folder_variant"] == variant,
        ["IMG_ID", "artery_dice", "ostia_detection_status"],
    ].rename(columns={"artery_dice": "variant_dice", "ostia_detection_status": "variant_ostia_status"})
    merged = baseline.merge(current, on="IMG_ID", how="inner")
    merged["dice_delta"] = merged["variant_dice"] - merged["baseline_dice"]
    delta_rows.append({
        "folder_variant": variant,
        "variant_label": PRETTY_NAMES.get(variant, variant),
        "mean_delta": merged["dice_delta"].mean(),
        "median_delta": merged["dice_delta"].median(),
        "improved_ge_0_02": int((merged["dice_delta"] >= 0.02).sum()),
        "worse_le_minus_0_02": int((merged["dice_delta"] <= -0.02).sum()),
        "max_gain": merged["dice_delta"].max(),
        "max_loss": merged["dice_delta"].min(),
    })

delta_summary_df = pd.DataFrame(delta_rows).sort_values("mean_delta", ascending=False)

In [ ]:
display(delta_summary_df.round({
    "mean_delta": 4,
    "median_delta": 4,
    "max_gain": 4,
    "max_loss": 4,
}))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_delta = delta_summary_df.copy()
colors = ["#2ca02c" if value >= 0 else "#d62728" for value in plot_delta["mean_delta"]]
bars = ax.bar(plot_delta["variant_label"], plot_delta["mean_delta"], color=colors)

for bar, value in zip(bars, plot_delta["mean_delta"]):
    va = "bottom" if value >= 0 else "top"
    offset = 0.002 if value >= 0 else -0.002
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + offset,
        f"{value:+.4f}",
        ha="center",
        va=va,
        fontsize=9,
    )

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta médio de Dice vs Normal + RG", fontsize=12)
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35, labelsize=10)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("dice_delta_vs_normal_rg.png")
plt.show()

## Maiores Variações por Comparação

As próximas células comparam pares específicos de variantes. Em cada comparação, o delta é calculado como `comparação - referência`. Valores positivos indicam que a segunda variante melhorou o Dice em relação à primeira.

In [ ]:
PAIR_COMPARISONS = [
    ("normal_rg", "th_fuzzy_rg", "Threshold fuzzy vs normal, ambos com RG"),
    ("normal_fc", "th_fuzzy_fc", "Threshold fuzzy vs normal, ambos com FC"),
    ("normal_rg", "normal_fc", "FC vs RG com threshold normal"),
    ("th_fuzzy_rg", "th_fuzzy_fc", "FC vs RG com threshold fuzzy"),
]


def make_pair_delta(reference_variant: str, comparison_variant: str) -> pd.DataFrame:
    reference = results_df.loc[
        results_df["folder_variant"] == reference_variant,
        ["IMG_ID", "artery_dice", "ostia_detection_status", "artery_voxel_count"],
    ].rename(columns={
        "artery_dice": "reference_dice",
        "ostia_detection_status": "reference_ostia_status",
        "artery_voxel_count": "reference_artery_voxels",
    })
    comparison = results_df.loc[
        results_df["folder_variant"] == comparison_variant,
        ["IMG_ID", "artery_dice", "ostia_detection_status", "artery_voxel_count"],
    ].rename(columns={
        "artery_dice": "comparison_dice",
        "ostia_detection_status": "comparison_ostia_status",
        "artery_voxel_count": "comparison_artery_voxels",
    })
    pair_df = reference.merge(comparison, on="IMG_ID", how="inner")
    pair_df["dice_delta"] = pair_df["comparison_dice"] - pair_df["reference_dice"]
    pair_df["abs_delta"] = pair_df["dice_delta"].abs()
    pair_df["reference_variant"] = reference_variant
    pair_df["comparison_variant"] = comparison_variant
    return pair_df


def pair_summary(reference_variant: str, comparison_variant: str) -> pd.DataFrame:
    pair_df = make_pair_delta(reference_variant, comparison_variant)
    return pd.DataFrame([
        {
            "reference": PRETTY_NAMES.get(reference_variant, reference_variant),
            "comparison": PRETTY_NAMES.get(comparison_variant, comparison_variant),
            "n_images": len(pair_df),
            "mean_delta": pair_df["dice_delta"].mean(),
            "median_delta": pair_df["dice_delta"].median(),
            "comparison_better_ge_0_02": int((pair_df["dice_delta"] >= 0.02).sum()),
            "reference_better_ge_0_02": int((pair_df["dice_delta"] <= -0.02).sum()),
            "max_gain": pair_df["dice_delta"].max(),
            "max_loss": pair_df["dice_delta"].min(),
        }
    ])


def largest_pair_changes(reference_variant: str, comparison_variant: str, top_n: int = 15) -> pd.DataFrame:
    pair_df = make_pair_delta(reference_variant, comparison_variant)
    return pair_df.sort_values("abs_delta", ascending=False).head(top_n)[[
        "IMG_ID",
        "reference_dice",
        "comparison_dice",
        "dice_delta",
        "reference_ostia_status",
        "comparison_ostia_status",
        "reference_artery_voxels",
        "comparison_artery_voxels",
    ]]


def plot_largest_pair_changes(reference_variant: str, comparison_variant: str, title: str, filename: str, top_n: int = 15):
    pair_df = make_pair_delta(reference_variant, comparison_variant)
    plot_df = pair_df.sort_values("abs_delta", ascending=False).head(top_n).sort_values("dice_delta")
    colors = ["#2ca02c" if value >= 0 else "#d62728" for value in plot_df["dice_delta"]]
    fig, ax = plt.subplots(figsize=(12, 5.5))
    bars = ax.barh(plot_df["IMG_ID"].astype(str), plot_df["dice_delta"], color=colors)
    for bar, value in zip(bars, plot_df["dice_delta"]):
        ha = "left" if value >= 0 else "right"
        offset = 0.003 if value >= 0 else -0.003
        ax.text(value + offset, bar.get_y() + bar.get_height() / 2, f"{value:+.3f}", va="center", ha=ha, fontsize=9)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Delta Dice: comparação - referência", fontsize=12)
    ax.set_ylabel("IMG_ID", fontsize=12)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    save_current_figure(filename)
    plt.show()

### Threshold fuzzy vs normal, ambos com RG

In [ ]:
display(pair_summary("normal_rg", "th_fuzzy_rg").round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes("normal_rg", "th_fuzzy_rg", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    "normal_rg",
    "th_fuzzy_rg",
    "Threshold fuzzy vs normal, ambos com RG",
    "largest_changes_th_fuzzy_rg_vs_normal_rg.png",
    top_n=15,
)

### Threshold fuzzy vs normal, ambos com FC

In [ ]:
display(pair_summary("normal_fc", "th_fuzzy_fc").round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes("normal_fc", "th_fuzzy_fc", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    "normal_fc",
    "th_fuzzy_fc",
    "Threshold fuzzy vs normal, ambos com FC",
    "largest_changes_th_fuzzy_fc_vs_normal_fc.png",
    top_n=15,
)

### FC vs RG com threshold normal

In [ ]:
display(pair_summary("normal_rg", "normal_fc").round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes("normal_rg", "normal_fc", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    "normal_rg",
    "normal_fc",
    "FC vs RG com threshold normal",
    "largest_changes_normal_fc_vs_normal_rg.png",
    top_n=15,
)

### FC vs RG com threshold fuzzy

In [ ]:
display(pair_summary("th_fuzzy_rg", "th_fuzzy_fc").round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes("th_fuzzy_rg", "th_fuzzy_fc", top_n=15).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    "th_fuzzy_rg",
    "th_fuzzy_fc",
    "FC vs RG com threshold fuzzy",
    "largest_changes_th_fuzzy_fc_vs_th_fuzzy_rg.png",
    top_n=15,
)

## Melhor RG vs Melhor FC

Esta comparação seleciona automaticamente a melhor variante RG e a melhor variante FC pelo ranking geral e mostra onde o FC mais ganha ou perde em relação ao RG.

In [ ]:
def best_variant_by_suffix(suffix: str) -> str:
    candidates = ranking_df[ranking_df["folder_variant"].astype(str).str.endswith(suffix)]
    if candidates.empty:
        raise ValueError(f"Nenhuma variante encontrada com sufixo {suffix}")
    return candidates.iloc[0]["folder_variant"]

best_rg_variant = best_variant_by_suffix("_rg")
best_fc_variant = best_variant_by_suffix("_fc")
print(f"Melhor RG: {PRETTY_NAMES.get(best_rg_variant, best_rg_variant)}")
print(f"Melhor FC: {PRETTY_NAMES.get(best_fc_variant, best_fc_variant)}")

In [ ]:
display(pair_summary(best_rg_variant, best_fc_variant).round({"mean_delta": 4, "median_delta": 4, "max_gain": 4, "max_loss": 4}))

In [ ]:
display(largest_pair_changes(best_rg_variant, best_fc_variant, top_n=20).round({"reference_dice": 4, "comparison_dice": 4, "dice_delta": 4}))

In [ ]:
plot_largest_pair_changes(
    best_rg_variant,
    best_fc_variant,
    "Melhor FC vs melhor RG",
    "largest_changes_best_fc_vs_best_rg.png",
    top_n=20,
)